# 05 — Rolling calibration and portfolio evaluation

Evaluate 1-, 5-, and 20-day forecasts across genuine forecast origins. Portfolio returns are computed correctly by converting each asset's log return to a simple return, applying weights, and compounding through time.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
CACHE = ROOT / 'results/notebook_cache'
CACHE.mkdir(parents=True, exist_ok=True)
from innovcal.evaluation.rolling import evaluate_rolling_forecasts, evaluate_rolling_portfolios

In [2]:
MODELS = ('gaussian', 'student_t', 'bootstrap', 'block_bootstrap', 'volatility_bootstrap', 'diffusion', 'cdi_var')
forecast_tensors = {}
observations = None
for model in MODELS:
    path = CACHE / f'rolling_{model}.npz'
    if path.exists():
        data = np.load(path)
        forecast_tensors[model] = data['forecasts']
        if observations is None:
            observations = data['observations']

if observations is None:
    raise FileNotFoundError('Run notebooks 03 and 04 before notebook 05.')
print('Loaded:', ', '.join(forecast_tensors))

Loaded: gaussian, student_t, bootstrap, block_bootstrap, volatility_bootstrap, diffusion, cdi_var


In [3]:
rolling_evaluation = evaluate_rolling_forecasts(
    forecast_tensors,
    observations,
    horizons=(1, 5, 20),
)
display(rolling_evaluation.sort_values(['horizon', 'energy_score']))

,innovation_model,horizon,n_origins,avg_coverage,avg_width,ece,pit_deviation,crps,energy_score,interval_score,abs_coverage_error
9,block_bootstrap,1,47,0.946809,0.055378,0.082270,0.028085,0.006943,0.016720,0.064889,0.046809
6,bootstrap,1,47,0.952128,0.054662,0.084043,0.025745,0.006972,0.016755,0.061962,0.052128
18,cdi_var,1,47,0.914894,0.046844,0.046809,0.024468,0.007021,0.016840,0.058677,0.014894
15,diffusion,1,47,0.957447,0.055892,0.112411,0.037234,0.007126,0.017090,0.063562,0.057447
12,volatility_bootstrap,1,47,0.968085,0.060463,0.124823,0.039362,0.007155,0.017147,0.066699,0.068085
3,student_t,1,47,0.952128,0.058848,0.110638,0.038298,0.007179,0.017187,0.065942,0.052128
0,gaussian,1,47,0.973404,0.062690,0.146099,0.049787,0.007470,0.017788,0.068399,0.073404
10,block_bootstrap,5,47,0.925532,0.055262,0.046809,0.018511,0.007697,0.018530,0.065287,0.025532
7,bootstrap,5,47,0.930851,0.056150,0.046809,0.019362,0.007698,0.018533,0.066381,0.030851
19,cdi_var,5,47,0.914894,0.048257,0.030851,0.017021,0.007762,0.018653,0.063509,0.014894


In [4]:
portfolio_evaluation = evaluate_rolling_portfolios(
    forecast_tensors,
    observations,
    weights=np.full(observations.shape[-1], 1 / observations.shape[-1]),
)
display(portfolio_evaluation.sort_values('portfolio_mae'))

,innovation_model,horizon,n_origins,portfolio_coverage,portfolio_interval_width,portfolio_mae,portfolio_var_breach_rate
3,block_bootstrap,20,47,0.978723,0.185935,0.029988,0.000000
0,gaussian,20,47,0.978723,0.184813,0.031405,0.021277
2,bootstrap,20,47,0.957447,0.181306,0.031430,0.042553
1,student_t,20,47,0.936170,0.185800,0.031638,0.063830
4,volatility_bootstrap,20,47,1.000000,0.184281,0.032283,0.000000
6,cdi_var,20,47,0.893617,0.126918,0.033989,0.042553
5,diffusion,20,47,0.914894,0.156579,0.034842,0.042553


In [5]:
rolling_evaluation.to_csv(CACHE / 'rolling_evaluation.csv', index=False)
portfolio_evaluation.to_csv(CACHE / 'portfolio_evaluation.csv', index=False)